# 线性拟合

线性拟合算是最为基础的机器学习模型，本文使用PyTorch的神经网络模块`torch.nn`搭建一个简单的线性拟合模型，并进行学习。

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
print(f'Import PyTorch V{torch.__version__}')

Import PyTorch V2.1.1


In [2]:
dev = torch.device(type='cuda') if torch.cuda.is_available() else torch.device(type='cpu')
print(f'Use device {dev}')

Use device cuda


## 模型

线性拟合的模型如下所示：

$$y = x A^T + b$$

其中$x$为输入数据，$y$为输出数据，$A$和$b$是需要拟合的模型参数，
令$d_i$和$d_o$分别代表输入和输出数据的维度，则$A$维度为$d_o \times d_i$，$b$维度为$d_o$。

这里假设问题的输入、输出维度分别为4和2，正确的解如下所示。

In [3]:
d_i, d_o = 4, 1
A_hat = torch.tensor([
    [1.2, 3.0, 7.7, -0.9],
])
b_hat = torch.tensor([0.7])

A_hat, b_hat

(tensor([[ 1.2000,  3.0000,  7.7000, -0.9000]]), tensor([0.7000]))

创建初始的模型

In [4]:
layer = torch.nn.Linear(d_i, d_o, device=dev)
model = torch.nn.Sequential(layer)
model

Sequential(
  (0): Linear(in_features=4, out_features=1, bias=True)
)

In [5]:
layer.weight, layer.bias

(Parameter containing:
 tensor([[ 0.3992,  0.3911, -0.2692, -0.4193]], device='cuda:0',
        requires_grad=True),
 Parameter containing:
 tensor([0.3445], device='cuda:0', requires_grad=True))

## 数据准备

随机生成1000个样本数据。

In [6]:
num_samples = 1000
X = torch.rand(num_samples, d_i) * 10.0
y = torch.matmul(X, A_hat.T) + b_hat + torch.normal(0.0, 0.01, (num_samples, d_o))
X = X.to(device=dev)
y = y.to(device=dev)
X.shape, y.shape

(torch.Size([1000, 4]), torch.Size([1000, 1]))

构造数据集，并以10为单位对数据进行分批。

In [7]:
batch_size = 10

ds = torch.utils.data.TensorDataset(X, y)
data_iter = torch.utils.data.DataLoader(ds, batch_size, shuffle=True)

next(iter(data_iter))

[tensor([[7.4849, 3.1878, 4.1681, 2.7792],
         [9.9061, 8.7683, 1.0625, 9.7647],
         [2.2392, 6.4695, 0.6913, 2.7427],
         [3.1500, 9.1339, 5.2870, 6.7559],
         [9.2232, 5.9165, 7.9823, 4.2015],
         [4.1182, 2.9701, 7.6940, 7.7901],
         [1.2023, 9.4200, 5.3274, 0.0796],
         [0.6525, 6.1053, 0.5265, 0.6027],
         [2.4961, 8.3309, 5.3297, 0.0974],
         [0.4863, 5.4129, 4.7647, 0.6925]], device='cuda:0'),
 tensor([[48.8223],
         [38.2761],
         [25.6406],
         [66.5239],
         [87.2028],
         [66.7741],
         [71.3457],
         [23.2873],
         [69.6409],
         [53.5999]], device='cuda:0')]

## 训练

首先，准备好损失函数，这里选择平均方差，相当于最小二乘。

In [8]:
loss = torch.nn.MSELoss()
loss

MSELoss()

接下来，准备训练器。

In [9]:
trainer = torch.optim.SGD(model.parameters(), lr=0.005)
trainer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    lr: 0.005
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

In [10]:
max_epochs = 1000
threshold = 2e-4
for epoch in range(max_epochs):
    for features, labels in data_iter:
        l = loss(model(features), labels)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(model(X), y)
    print(f'epoch {epoch + 1}, loss {l:f}')
    if l < threshold:
        break

epoch 1, loss 0.001647
epoch 2, loss 0.001463
epoch 3, loss 0.001184
epoch 4, loss 0.001379
epoch 5, loss 0.000884
epoch 6, loss 0.000772
epoch 7, loss 0.000682
epoch 8, loss 0.000713
epoch 9, loss 0.000530
epoch 10, loss 0.000451
epoch 11, loss 0.000556
epoch 12, loss 0.000399
epoch 13, loss 0.000325
epoch 14, loss 0.000300
epoch 15, loss 0.000261
epoch 16, loss 0.000240
epoch 17, loss 0.000225
epoch 18, loss 0.000285
epoch 19, loss 0.000239
epoch 20, loss 0.000172


In [11]:
layer.weight, layer.bias

(Parameter containing:
 tensor([[ 1.1986,  2.9983,  7.6989, -0.9010]], device='cuda:0',
        requires_grad=True),
 Parameter containing:
 tensor([0.7285], device='cuda:0', requires_grad=True))

In [12]:
A_hat, b_hat

(tensor([[ 1.2000,  3.0000,  7.7000, -0.9000]]), tensor([0.7000]))